# A QC report, built by API instead of by automation

The app ships a **Quality Control Report** automation. You pick it from the Automations menu, it
asks four questions, and it lays out a full QC report on the tab.

This notebook does the same thing from the API. Same modules, same grid positions, same settings,
no clicking. Which means you can attach it to the end of a pipeline and have the report waiting when
the data lands.

### What the in-app automation actually is

Automations are declarative workflows that run **in the browser**
(`workflow/app/javascript/workspaces/lib/automations/QCReport.js`). There is no automations endpoint
on the API, so you cannot trigger one remotely. But you do not need to: an automation is a list of
`AddModuleToTab` steps, and adding a module to a tab is a plain API call.

The real automation asks for four things once:

| it asks | we set once, below |
|---|---|
| `datasetsSearch` | the intensity dataset |
| `entityType` | Protein or Peptide |
| `groupBy` | the metadata column to colour and group by |
| `xAxis` | the metadata column to order samples along |

then places **29 modules** using them. This notebook does exactly that.

## Setup

In [ ]:
import os

from dotenv import load_dotenv
from md_python import MDClient

load_dotenv()
assert os.getenv("MD_AUTH_TOKEN"), "MD_AUTH_TOKEN is not set (check your .env)"

client = MDClient(version="v2")
print("API:", client.base_url)

## Configuration

The four answers the automation would ask for, plus where to put the report.

### `GROUP_BY` and `X_AXIS` are sample metadata column names

These two are not free text and they are not settings of the module. They are the names of columns
in the **sample metadata of the upload the dataset came from**, and almost every data module on the
tab reads one of them:

| module | setting | value |
|---|---|---|
| dimensionality reduction (x5) | `colourBy` | `GROUP_BY` |
| CV distribution, CV violin | `xAxis` | `GROUP_BY` |
| box plot | `xAxis` | `X_AXIS` |
| intensity distribution | `groupByColumn` | `GROUP_BY` |
| QC summary table (x3) | `xAxis`, labelled Table Grouping | `GROUP_BY` |

For this dataset the grouping column is `condition`, so that is what `GROUP_BY` is set to below.
`sample_name` is the one column you can rely on always being there, which is why it is the default
for ordering.

Nothing checks these for you. The registry cannot publish a list of valid values, because the valid
values are whatever columns that particular upload happens to carry, so a column name that does not
exist is stored without complaint and the module comes back empty. This is the same class of problem
as the empty Table Grouping described in section 3.

**What that means if you are automating this.** A script like this one is only reusable across
uploads if those uploads share a sample metadata schema. If every upload you produce carries a
column called `condition`, this notebook runs unchanged on all of them. If one pipeline writes
`condition`, another writes `treatment` and a third writes `Condition`, then the same script builds
a report of blank tiles on two of the three and tells you nothing went wrong. Deciding on the
metadata column names once, upstream at upload time, is what makes the report automatable at all.

In [ ]:
# The intensity dataset to report on.
DATASET_ID = "00000000-0000-0000-0000-000000000000"

# LOWERCASE. protein | peptide | gene | metabolite | ptm
# Capitalised values are accepted by the API and then fail to render.
ENTITY_TYPE = "protein"

# Sample metadata columns of the upload behind DATASET_ID, not module settings.
# GROUP_BY colours and groups; X_AXIS orders the samples. A name that is not a
# real column is accepted and renders empty, so check these against your upload.
GROUP_BY = "condition"
X_AXIS = "sample_name"

# Scaling for the dimensionality reduction and heatmap. none | zscore | centered
# The in-app automation uses zscore. Leaving this at the "none" default is
# almost never what you want: unscaled PCA is dominated by the most abundant
# proteins.
SCALING_METHOD = "zscore"

WORKSPACE_NAME = "QC report (API)"
TAB_NAME = "Quality control"

## 1. A workspace and a tab to put it on

Skip this if you already have one: set `WORKSPACE_ID` and `TAB_ID` yourself.

In [ ]:
workspace = client.workspaces.create(
    name=WORKSPACE_NAME,
    description="QC report generated through the API",
)
WORKSPACE_ID = str(workspace.id)

tab = client.workspaces.tabs.create(workspace_id=WORKSPACE_ID, name=TAB_NAME)
TAB_ID = str(tab.id)

print("workspace:", WORKSPACE_ID)
print("tab      :", TAB_ID)

## 2. The shared settings

Every data module in the report gets the same dataset and entity type. This is the automation's
"ask once, use everywhere" made literal.

`datasetsSearch` is a structured field: it needs `{"individualResults": [{"id", "name"}]}` with
**both** id and name, not a bare id.

In [ ]:
dataset = client.datasets.get_by_id(DATASET_ID)
assert dataset, f"No dataset {DATASET_ID!r}"
print(f"{dataset.name}  ({dataset.type}, {dataset.state})")

COMMON = {
    "datasetsSearch": {
        "individualResults": [{"id": DATASET_ID, "name": dataset.name}]
    },
    "entityType": ENTITY_TYPE,
}

X_AXIS_ORDERED = [{"field": X_AXIS, "order": "none"}]

# Grouping by condition, in the shape the automation uses everywhere it groups:
# the CV plots and the QC summary tables, where it is labelled Table Grouping.
# Two things to note. The summary table's grouping key is called `xAxis`, and
# its registry default is an empty list, so a table placed without it comes out
# blank. And the CV plots must be grouped by GROUP_BY rather than by X_AXIS: a
# coefficient of variation needs at least two replicates per group, so grouping
# by sample_name gives one sample per group and the render fails outright.
GROUPING = [{"field": GROUP_BY, "order": "asc", "type": "categorical"}]

# Fetch the registry once; the validator below reads it.
REGISTRY = {str(m.id): m for m in client.module_registry.list()}

## 3. Validate before you build

The API will accept settings that do not work. A value outside a field's allowed list is stored
happily and only shows up later as an empty dropdown in the app and a 500 from the visualisation
endpoint. That is what broke the first version of this notebook: `entityType` was sent as
`"Protein"` instead of `"protein"`.

Three things are checkable up front:

| check | how |
|---|---|
| keys the module does not declare | `module.validate_settings_keys(settings)` |
| required keys with no value | `module.missing_required_keys({**defaults, **settings})` |
| values outside an allowed list | `spec["parameters"]["options"]` |

That third one is the important one and it is easy to miss: **142 of the 625 settings across the
registry publish an options list**, under `parameters.options`, not at the top level of the spec.

There is a fourth mistake none of the three catch: a **required key whose default is empty**.
`qc_summary_table`'s Table Grouping (`xAxis`) is required, and the registry default is `[]`. An
empty list is a value, so `missing_required_keys` is satisfied, the module is created, and the table
renders with nothing to group by. Required and populated are not the same check.

`entityType` is the awkward exception. It publishes no options, because its valid values depend on
the dataset and the organisation's feature flags, so you have to know them:
`protein`, `peptide`, `gene`, `metabolite`, `ptm`, all lowercase.

In [ ]:
# Entity types, lowercase. The registry does NOT publish these as options
# (fieldType "EntityType" is resolved from the dataset at render time), so this
# is the one list you have to know rather than look up.
# Source: workflow app/javascript/workspaces/lib/helpers/EntityListHelper.js
ENTITY_TYPES = {"protein", "peptide", "gene", "metabolite", "ptm"}


def check_settings(item_id, settings):
    """Return a list of problems with `settings` for `item_id`, empty if fine.

    Catches three classes of mistake before anything is sent:

      1. keys the module does not declare      (server rejects these outright)
      2. required keys with no value            (module renders broken)
      3. values outside a declared options list (silently accepted, then fails)

    (3) is the dangerous one. The API accepts any string for an enum field, so a
    wrong value is only visible as an empty dropdown in the app and a 500 from
    the visualisation endpoint.
    """
    module = REGISTRY.get(item_id)
    if module is None:
        return [f"{item_id!r} is not in the module registry"]

    problems = []
    schema = module.input_settings or {}
    specs = schema if isinstance(schema, dict) else {
        s.get("key"): s for s in schema if isinstance(s, dict)
    }

    unknown = module.validate_settings_keys(settings)
    if unknown:
        problems.append(f"keys not declared by this module: {list(unknown)}")

    missing = module.missing_required_keys({**module.defaults(), **settings})
    if missing:
        problems.append(f"required with no value: {missing}")

    for key, value in settings.items():
        spec = specs.get(key)
        if not isinstance(spec, dict):
            continue
        options = (spec.get("parameters") or {}).get("options")
        if options:
            allowed = [o.get("value") for o in options if isinstance(o, dict)]
            if allowed and value not in allowed:
                problems.append(f"{key}={value!r} not one of {allowed}")

    if settings.get("entityType") not in (None, *ENTITY_TYPES):
        problems.append(
            f"entityType={settings['entityType']!r} is not one of "
            f"{sorted(ENTITY_TYPES)} (lowercase)"
        )
    return problems


# Prove it catches the mistake that broke this notebook the first time.
bad = {**COMMON, "entityType": "Protein"}
print("Checking a deliberately wrong entityType:")
for p in check_settings("dimensionality_reduction_plot", bad):
    print("   ", p)
print("\nChecking the real settings:",
      check_settings("dimensionality_reduction_plot", COMMON) or "no problems")

## 4. The report, as data

The automation is a chain of builder calls. Here the same thing is a list: one row per module, with
its grid position and whatever it needs on top of `COMMON`.

Positions are taken from the automation itself, so the result matches what the app produces. The
grid is 12 columns wide, and `y` runs down the page.

Because reading 29 rows of chained builder code is nobody's idea of fun, this is where the API
version is arguably nicer than the automation: the report is a table you can edit, reorder, or
generate.

In [ ]:
def note(title, body):
    """Workspace text modules take PLAIN TEXT. HTML tags are not allowed."""
    return {"text": f"{title}\n{body}"}


# Settings shared by the five dimensionality reduction plots. The in-app
# automation sets scaling and colours by the grouping column rather than
# leaving the sample_name default.
DR = {"scalingMethod": SCALING_METHOD, "colourBy": GROUP_BY, "drMethod": "pca"}

REPORT = [
    # item_id,                         x, y,   w,  h,  extra settings
    ("heading",                        0,   0, 12,  3, {"text": "Quality Control Report"}),
    ("text",                           0,   3, 12,  6,
     note("Dimensionality reduction",
          "How samples relate to one another. Replicates should cluster.")),

    # Variance explained, then PC1 against PC2, PC3, PC4, PC5.
    ("dimensionality_reduction_plot",  0,   9,  6, 20, {**DR, "drSubtype": "pca_variance"}),
    ("dimensionality_reduction_plot",  6,   9,  6, 20,
     {**DR, "drSubtype": "pca", "nComponents": {"xAxis": "PC1", "yAxis": "PC2"}}),
    ("dimensionality_reduction_plot",  0,  29,  4, 12,
     {**DR, "drSubtype": "pca", "nComponents": {"xAxis": "PC1", "yAxis": "PC3"}}),
    ("dimensionality_reduction_plot",  4,  29,  4, 12,
     {**DR, "drSubtype": "pca", "nComponents": {"xAxis": "PC1", "yAxis": "PC4"}}),
    ("dimensionality_reduction_plot",  8,  29,  4, 12,
     {**DR, "drSubtype": "pca", "nComponents": {"xAxis": "PC1", "yAxis": "PC5"}}),

    ("text",                           0,  41, 12,  5,
     note("Variability", "Coefficient of variation within each group.")),
    ("cv_distribution_violin_plot",    0,  46, 12, 15, {"xAxis": GROUPING}),
    ("cv_distribution_plot",           0,  61, 12, 14, {"xAxis": GROUPING}),
    ("qc_summary_table",               0,  75, 12, 15, {"xAxis": GROUPING}),

    ("text",                           0,  90, 12,  5,
     note("Sample correlation", "Correlation between samples, clustered.")),
    # The heatmap has no scalingMethod. Its equivalent is normalisationMethod,
    # which already defaults to zscore. The QC report uses it as a sample
    # CORRELATION heatmap, which is a different dataType.
    ("heatmap",                        0,  95, 12, 21,
     {"dataType": "correlation", "correlationAxis": "samples",
      "correlationMethod": "pearson", "showSampleNamesCorrelationHeatmap": True}),

    ("text",                           0, 116, 12,  5,
     note("Missing values", "Where the gaps are, by feature and by sample.")),
    ("missing_values_by_feature_plot", 0, 121,  6, 19, {}),
    ("missing_values_by_sample_plot",  6, 121,  6, 19, {}),
    ("missing_values_heatmap",         0, 140, 12, 25, {}),
    ("qc_summary_table",               0, 165, 12, 15, {"xAxis": GROUPING}),

    ("text",                           0, 180, 12,  5,
     note("Intensity distributions", "Per-sample intensity spread.")),
    ("box_plot",                       0, 185, 12, 16, {"xAxis": X_AXIS_ORDERED}),
    ("intensity_distribution_plot",    0, 201, 12, 15, {"groupByColumn": GROUP_BY}),
    ("qc_summary_table",               0, 216, 12, 15, {"xAxis": GROUPING}),
]

print(f"{len(REPORT)} modules, {max(y + h for _, _, y, _, h, _ in REPORT)} grid rows tall")

## 5. Build it

One loop. `create_with_defaults()` fills in every registry default, so each row only carries what
differs from the default.

If a module rejects the settings it is reported and skipped rather than aborting the run, so you get
as much of the report as the data supports.

In [ ]:
LAYOUT_ONLY = {"heading", "text"}

# Validate everything BEFORE creating anything. A bad enum value is accepted by
# the API and only shows up as a broken widget, so catching it here is the
# difference between a report and a page of error tiles.
problems = []
for item_id, x, y, w, h, extra in REPORT:
    settings = dict(extra) if item_id in LAYOUT_ONLY else {**COMMON, **extra}
    for p in check_settings(item_id, settings):
        problems.append(f"{item_id} (y={y}): {p}")

if problems:
    print(f"{len(problems)} problem(s) found, nothing was created:\n")
    for p in problems:
        print("   ", p)
    raise SystemExit("Fix the settings above and re-run.")

print("All settings validate. Building.\n")

placed = []
for item_id, x, y, w, h, extra in REPORT:
    settings = dict(extra) if item_id in LAYOUT_ONLY else {**COMMON, **extra}
    module = client.workspaces.modules.create_with_defaults(
        workspace_id=WORKSPACE_ID, tab_id=TAB_ID, item_id=item_id,
        x=x, y=y, width=w, height=h, settings=settings,
    )
    placed.append(module)
    print(f"  ok  {item_id:<32} x={x:<3} y={y:<4} w={w:<3} h={h}")

print(f"\n{len(placed)} modules placed")

## 6. What ended up on the tab

In [ ]:
for m in sorted(client.workspaces.modules.list(WORKSPACE_ID, TAB_ID),
                key=lambda m: (m.y, m.x)):
    print(f"  {str(m.item_id):<32} x={m.x:<3} y={m.y:<4} w={m.width:<3} h={m.height}")

print(f"\nOpen it: {client.base_url.replace('/api', '')}/w/{WORKSPACE_ID}/dashboard")

## 7. Pull a figure back out

The report is now in the app. If you also want the figures headlessly, render any module to its
Plotly payload and write it out. There is no server-side image export, so rendering to PNG happens
locally.

```python
result = client.workspaces.modules.visualize(WORKSPACE_ID, TAB_ID, module_id)
# result == {"data": [...], "layout": {...}}

import plotly.io as pio
pio.write_image(result, "figure.png")   # needs kaleido
```

In [ ]:
# Not every module renders server-side, and some currently error. Report the
# reason per module and keep going: the point is the report, which is already
# built and visible in the app either way.
candidates = [m for m in placed
              if str(m.item_id) not in {"heading", "text", "qc_summary_table"}]

rendered = None
for target in candidates:
    try:
        result = client.workspaces.modules.visualize(
            WORKSPACE_ID, TAB_ID, str(target.id), poll_s=3, timeout_s=120
        )
    except Exception as exc:
        reason = "no server-side render" if "not supported" in str(exc) else \
                 f"error: {str(exc).splitlines()[0][:70]}"
        print(f"  {str(target.item_id):<32} {reason}")
        continue

    if isinstance(result, dict) and "data" in result:
        rendered = (target, result)
        break
    print(f"  {str(target.item_id):<32} still rendering")

if rendered:
    target, result = rendered
    print(f"\nRendered {target.item_id}: {len(result['data'])} trace(s)")
    for t in result["data"][:4]:
        print(f"    {t.get('type', '?'):<10} {str(t.get('name')):<26} "
              f"{len(t.get('x', []) or [])} points")
else:
    print("\nNothing rendered headlessly this time. The report itself is fine "
          "and is visible in the app.")

## Making it a real automation

Nothing here needs a notebook. Drop sections 2 to 4 into a `.py` file, take `DATASET_ID` from an
argument or an environment variable, and call it at the end of your pipeline:

```python
build_qc_report(dataset_id=sys.argv[1])
```

The report exists before anyone opens the app. That is the part the in-app automation cannot do:
it needs a person and a browser.

Two honest limits:

* **You cannot trigger an in-app automation over the API.** They are frontend workflows. What you
  can do is reproduce their output, which is what this notebook is.
* **`quality_control_report_classic`** is a single module that renders an entire QC report on its
  own, and it only needs `{"experimentId": <upload id>}`. If that is all you want, place that one
  module instead of the 22 here. This notebook builds the report out of individual modules because
  that is what the modern automation does, and because it is the version you can edit.